# Stage 2 Notebook 43 - Exp2NN Anchor + mask-consistency decode + cls self-distill

**The biggest leap so far: drop the cls task from the decode bottleneck.** Across NB39 (focal), NB40 (ASL), NB41 (lineiou regression+QFL) and NB42 (dual cls+iou), the cls head's pos-vs-neg score gap stayed within 0.002 to 0.01 -- the cls is performing at chance level on the 192-anchor head. Geometry is excellent (matched_iou=0.508, oracle_f1=0.459 in NB41) but decoded_f1 plateaued at 0.04 because we cannot rank the 192 priors by the cls head's output.

Exp2NN replaces cls-based ranking entirely:

1. **Inference-side**: `eval.decoded_score_source = cls_x_mask`. For each prior, sample the auxiliary mask sigmoid along the predicted curve and use the mean as a ranking score. The aux mask is trained on per-pixel BCE+Dice with no per-prior matching instability, so its sigmoid is a clean per-prior geometric verifier. Hybrid score = sigmoid(cls) * mask_consistency keeps any cls signal that does separate priors.
2. **Training-side**: `cls_target_type = mask_consistency`. cls is self-distilled to predict the same mask-along-curve score. This gives cls a stable, deterministic, batch-stable supervision target -- the antidote to the matching instability that has been collapsing it.
3. **Companion diagnostics**: the train script automatically reports `decoded_cls_only_f1` and `decoded_mask_only_f1` alongside the primary `decoded_f1`. So one notebook produces a clean three-way comparison of cls / mask / cls*mask rankings.

Code changes (new since NB42):
- `losses.py`: new `_compute_mask_consistency_target` plus a `cls_target_type='mask_consistency'` branch in `_forward_single_stage` that supervises cls with the mask-along-curve score (BCE or QFL).
- `metrics/lane_f1_decoded.py`: new `score_source='mask_consistency'` and `'cls_x_mask'`. `_mask_consistency_score` samples the head's `mask_logit` along each prior's `coord_pred`.
- `train_joint_model_experiment.py`: `_cls_only` and `_mask_only` companion metrics auto-fire when the primary score source is a hybrid.

Reference: this is conceptually similar to RTMDet's centerness * cls product, except we replace the cls with a segmentation-derived score that is robust to per-prior label noise.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run -- the new mask-consistency code path needs a smoke check.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. AMP keeps wall-clock ~ 30 minutes for 20 epochs at 3000 samples.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00. Independent of NB35 / NB39-42; only depends on the dataset tar.

In [ ]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp38_rmt_gca_anchor_mask_consistency_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp38_rmt_gca_anchor_mask_consistency_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

## What to watch in Exp2NN training

Reference NB41 (anchor + lineiou regression + QFL): matched_iou=0.508, oracle_f1=0.459, decoded_f1=0.029, pos-neg gap = 0.002.
Reference NB42 (anchor + dual cls*iou): matched_iou=0.490, oracle_f1=0.426, decoded_f1=0.042 (cls*iou), decoded_cls_only_f1=0.012.

Pass criteria at epoch 20:
- **`val/lane/decoded_f1 >= 0.20`** (cls * mask). 5x NB42, 7x NB41. The mask-consistency score should actually rank priors meaningfully.
- **`val/lane/decoded_mask_only_f1 >= 0.18`**. The mask alone (no cls) should already give most of the gain.
- **`val/lane/decoded_cls_only_f1 >= 0.05`**. cls is now self-distilled to mimic the mask, so it should ALSO rank meaningfully (3x its NB42 value), proving the supervision target was the bottleneck.
- **`val/matched_line_iou >= 0.45`** (preserve geometric champion).
- **`val/lane/decoded_oracle_f1 >= 0.40`** (oracle ceiling stays high).

Failure signals:
- mask_only_f1 ~ cls_only_f1 ~ 0.05: the mask is also not discriminative on this dataset. Inspect val_lane_mask trend (should fall from 0.55 to ~ 0.40 over training).
- decoded_cls_only_f1 << decoded_mask_only_f1: self-distillation didn't take. Bump `w_cls` to 7.0 or switch `cls_loss_type: qfl`.
- matched_iou drops below 0.40: the mask supervision is somehow distorting geometry. Lower `w_mask` to 1.0.

If decoded_f1 jumps to >= 0.20, the cls collapse on the anchor head was the structural bottleneck and Exp2NN is the new champion. The mask is geometry-aware and free of the matching-instability.